<a href="https://colab.research.google.com/github/Galaticos402/AddressSegmentation/blob/main/Copy_of_GettingStarted_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
# Change your team name to Group_xx
TEAM_NAME = 'Group_02'

In [62]:
# NOTE: you CAN change this cell
# If you want to use your own database, download it here
# !gdown ...

In [63]:
# NOTE: you CAN change this cell
# Add more to your needs
# you must place ALL pip install here
!pip install editdistance

In [64]:
# NOTE: you CAN change this cell
# import your library here
import time
import unicodedata
import re
import editdistance

In [65]:
class Node:
    def __init__(self, value: str, is_terminated = False):
        self.val = value
        self.children = {}
        self.is_terminated = is_terminated
        self.original_word = None

class Trie:
    def __init__(self):
        self.root = Node(None)

    def insert(self, word: str, original_word: str = None):
        ite = self.root
        for c in word:
            if c not in ite.children:
                ite.children[c] = Node(c)
            ite = ite.children[c]
        ite.is_terminated = True
        ite.original_word = original_word

    def contains(self, word: str):
        ite = self.root
        for c in word:
            if c not in ite.children:
                return None
            ite = ite.children[c]
        if ite.is_terminated and ite.original_word is not None:
            return ite.original_word
        return None

    def searchByPrefix(self, prefix: str) -> list[str]:
        ite = self.root
        for c in prefix:
            if c not in ite.children:
                # Prefix not found in Trie, therefore return empty
                return []
            ite = ite.children[c]
        viable_options = self._dfs(ite)
        result = [prefix + word for word in viable_options]
        return result

    def getAllWords(self) -> list[str]:
        return self._dfs(self.root)

    def _dfs(self, node: Node) -> list[str]:
        result: list[str] = []
        stack: list[dict] = []

        stack.append({
            "node": node,
            "word": ""
        })

        while len(stack) > 0:
            item = stack.pop()

            current_node = item["node"]
            current_word = item["word"]

            if current_node.is_terminated:
                result.append(current_word)

            for letter, child in current_node.children.items():
                stack.append({
                    "node": child,
                    "word": current_word + letter
                })

        return result

In [66]:
from typing_extensions import Self
# NOTE: you MUST change this cell
# New methods / functions must be written under class Solution.
class Solution:
    def __init__(
        self,
        list_province: list[str],
        list_ward: list[str],
        list_street: list[str]
    ): # list province, ward, street
        self.list_province = list_province
        self.list_ward = list_ward
        self.list_street = list_street

        # write your preprocess here, add more method if needed
        self.street_dict = Trie()
        self.ward_dict = Trie()
        self.province_dict = Trie()
        self.typo_dict = {
            "0": ["o"],
            "o": ["0"],
            "1": ["l", "i"],
            "l": ["1"],
            "i": ["1"],
            "5": ["s"],
            "s": ["5"],
            "8": ["b"],
            "b": ["8"],
        }

        self.PROVINCE_PREFIXES = ["tp", "thanhpho", "tinh", "t"]
        self.PROVINCE_POSTFIXES = ["c", "city", "province"]
        self.WARD_PREFIXES = ["p", "phuong", "quan", "q"]
        self.WARD_POSTFIXES = ["ward", "w", "district"]
        self.STREET_PREFIXES = ["d", "duong"]
        self.STREET_POSTFIXES = ["street", "str", "st", "s"]
        self.COMMON_ABBREVIATION_DICT={
            "hochiminh": "hcm",
            "hanoi": "hn",
            "lythuongkiet": "ltk",
            "longan": "la",
            "angiang": "ag",
            "bariavungtau": "brvt",
            "cantho": "ct",
            "suvanhanh": "svh",
        }

        self.REVERSE_ABBREVIATION_DICT = {
            abbreviation: full_name for full_name, abbreviation in self.COMMON_ABBREVIATION_DICT.items()
        }

        self.reprocess()
        pass

    def normalize(self, sentence: str):
        sentence = sentence.lower()
        sentence = sentence.replace("Đ", "D").replace("đ", "d")
        sentence = unicodedata.normalize("NFD", sentence)
        sentence = "".join(c for c in sentence if unicodedata.category(c) != "Mn")
        sentence = re.sub(r"[^a-zA-Z0-9]", "", sentence)
        return sentence

    def possibleMatches(self, s: str, trie: Trie):
        matches = []

        size = len(s)

        for i in range(size + 1):
            for j in range(i):
                if trie.contains(s[j:i]) is not None:
                    matches.append({
                        "word": s[j:i],
                        "original_word": trie.contains(s[j:i]),
                    })
        print(f"Possible matches for {s}: {matches}")
        return list(matches)

    def getMostReasonableCombination(self, matches: list[dict], s: str):
        highest_match = matches[0] if matches else {"province": None, "ward": None, "street": None}
        highest_match_score = 0
        for match in matches:
            score = 0
            score = score + 1 if match["province"] is not None else score
            score = score + 1 if match["ward"] is not None else score
            score = score + 1 if match["street"] is not None else score
            if score > highest_match_score:
                highest_match_score = score
                highest_match = match
        return highest_match

    def generate_typos(self, s: str):
        typos = set()
        for i, char in enumerate(s):
            if char in self.typo_dict:
                for typo_char in self.typo_dict[char]:
                    typo = s[:i] + typo_char + s[i+1:]
                    typos.add(typo)
        return typos

    def generateVariants(self, s: str, type: str):
        if type == "PROVINCE":
            return self._generateProvinceVariants(s)
        if type == "WARD":
            return self._generateWardVariants(s)
        if type == "STREET":
            return self._generateStreetVariants(s)
        return [s]

    def _generateStreetVariants(self,street: str):
        variants = {street}
        if self.COMMON_ABBREVIATION_DICT.get(street) is not None:
            variants.add(self.COMMON_ABBREVIATION_DICT.get(street))

        if self.REVERSE_ABBREVIATION_DICT.get(street) is not None:
            variants.add(self.REVERSE_ABBREVIATION_DICT.get(street))

        for prefix in self.STREET_PREFIXES:
            if street.startswith(prefix):
                actual_street = street[len(prefix):]
                variants.add(actual_street)
                if self.COMMON_ABBREVIATION_DICT.get(actual_street) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_street))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_street) is not None:
                    print(f"Adding reverse abbreviation for street '{actual_street}': {self.REVERSE_ABBREVIATION_DICT.get(actual_street)}")
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_street))

        for postfix in self.STREET_POSTFIXES:
            if street.endswith(postfix):
                actual_street = street[:-len(postfix)]
                variants.add(actual_street)
                if self.COMMON_ABBREVIATION_DICT.get(actual_street) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_street))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_street) is not None:
                    print(f"Adding reverse abbreviation for street '{actual_street}': {self.REVERSE_ABBREVIATION_DICT.get(actual_street)}")
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_street))

        variants.update(
            pattern
            for variant in variants.copy()
            for pattern in (
                f"d{variant}",
                f"duong{variant}",
                f"{variant}street",
                f"{variant}st",
            )
        )
        print(f"Generated variants for street '{street}': {variants}")
        return variants


    def _generateWardVariants(self,ward: str):
        variants = {ward}
        if self.COMMON_ABBREVIATION_DICT.get(ward) is not None:
            variants.add(self.COMMON_ABBREVIATION_DICT.get(ward))

        for prefix in self.WARD_PREFIXES:
            if ward.startswith(prefix):
                actual_ward = ward[len(prefix):]
                variants.add(actual_ward)
                if self.COMMON_ABBREVIATION_DICT.get(actual_ward) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_ward))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_ward) is not None:
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_ward))

        for postfix in self.WARD_POSTFIXES:
            if ward.endswith(postfix):
                actual_ward = ward[:-len(postfix)]
                variants.add(actual_ward)
                if self.COMMON_ABBREVIATION_DICT.get(actual_ward) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_ward))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_ward) is not None:
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_ward))

        variants.update(
            pattern
            for variant in variants.copy()
            for pattern in (
                f"p{variant}",
                f"phuong{variant}",
                f"q{variant}",
                f"quan{variant}",
                f"{variant}ward",
                f"{variant}w",
            )
        )
        print(f"Generated variants for ward '{ward}': {variants}")
        return variants

    def _generateProvinceVariants(self, province: str):
        variants = {province}
        if self.COMMON_ABBREVIATION_DICT.get(province) is not None:
            variants.add(self.COMMON_ABBREVIATION_DICT.get(province))

        for prefix in self.PROVINCE_PREFIXES:
            if province.startswith(prefix):
                actual_province = province[len(prefix):]
                variants.add(actual_province)
                if self.COMMON_ABBREVIATION_DICT.get(actual_province) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_province))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_province) is not None:
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_province))

        for postfix in self.PROVINCE_POSTFIXES:
            if province.endswith(postfix):
                actual_province = province[:-len(postfix)]
                variants.add(actual_province)
                if self.COMMON_ABBREVIATION_DICT.get(actual_province) is not None:
                    variants.add(self.COMMON_ABBREVIATION_DICT.get(actual_province))

                if self.REVERSE_ABBREVIATION_DICT.get(actual_province) is not None:
                    variants.add(self.REVERSE_ABBREVIATION_DICT.get(actual_province))

        variants.update(
            pattern
            for variant in variants.copy()
            for pattern in (
                f"tp{variant}",
                f"thanhpho{variant}",
                f"tinh{variant}",
                f"t{variant}",
                f"{variant}city",
                f"{variant}c",
            )
        )
        print(f"Generated variants for province '{province}': {variants}")
        return variants

    def reprocess(self):
      for street in self.list_street:
        street_normalized = self.normalize(street)
        variants = self.generateVariants(street_normalized, "STREET")
        for v in variants:
            word = self.normalize(v[::-1])
            self.street_dict.insert(word, original_word=street)
            typos = self.generate_typos(word)
            for typo in typos:
                self.street_dict.insert(typo, original_word=street)

      for ward in self.list_ward:
        ward_normalized = self.normalize(ward)
        variants = self.generateVariants(ward_normalized, "WARD")
        for v in variants:
            word = self.normalize(v[::-1])
            self.ward_dict.insert(word, original_word=ward)
            typos = self.generate_typos(word)
            for typo in typos:
                self.ward_dict.insert(typo, original_word=ward)

      for province in self.list_province:
        province_normalized = self.normalize(province)
        variants = self.generateVariants(province_normalized, "PROVINCE")
        for v in variants:
            word = self.normalize(v[::-1])
            self.province_dict.insert(word, original_word=province)
            typos = self.generate_typos(word)
            for typo in typos:
                self.province_dict.insert(typo, original_word=province)

    def process(self, s: str):
        # write your process string here
        normalized_address = self.normalize(s[::-1])
        possible_provinces = self.possibleMatches(normalized_address, self.province_dict) or [None]
        combinations = []

        for province in possible_provinces:
          address_without_province = normalized_address

          if province is not None:
            address_without_province = normalized_address.replace(
              province["word"], "", 1
            )

          print(f"Address without province: {address_without_province}")

          possible_wards = self.possibleMatches(
            address_without_province, self.ward_dict
          ) or [None]

          for ward in possible_wards:
            address_without_ward = address_without_province

            if ward is not None:
              address_without_ward = address_without_province.replace(
                ward["word"], "", 1
              )

            print(f"Address without ward: {address_without_ward}")

            possible_streets = self.possibleMatches(
              address_without_ward, self.street_dict
            ) or [None]

            for street in possible_streets:
              combinations.append({
                "province": (
                    province["original_word"]
                    if province is not None else None
                ),
                "ward": (
                    ward["original_word"]
                    if ward is not None else None
                ),
                "street": (
                    street["original_word"]
                    if street is not None else None
                ),
              })

        most_reasonable_combination = self.getMostReasonableCombination(combinations, normalized_address)

        print("Most Reasonable Combination:")
        print(most_reasonable_combination)

        return {
            "province": most_reasonable_combination['province'],
            "ward": most_reasonable_combination['ward'],
            "street": most_reasonable_combination['street'],
        }

In [67]:
# Download public test
!rm -rf test.json
!gdown --fuzzy https://drive.google.com/file/d/18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW/view?usp=sharing -O test.json
!gdown --fuzzy https://drive.google.com/file/d/1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU/view?usp=drive_link -O list_province.txt
!gdown --fuzzy https://drive.google.com/file/d/1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h/view?usp=drive_link -O list_ward.txt
!gdown --fuzzy https://drive.google.com/file/d/1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa/view?usp=drive_link -O list_street.txt

Downloading...
From: https://drive.google.com/uc?id=18qzaHmx9-fcKPi4gTq304G2JvXy_g6XW
To: /content/test.json
100% 42.7k/42.7k [00:00<00:00, 70.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PHRkRLRaO2vFivLbcPJsaxjvZ7K_I3lU
To: /content/list_province.txt
100% 392/392 [00:00<00:00, 1.33MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QrSouYn6IUEW7Impje1vOxzsksjlAJ-h
To: /content/list_ward.txt
100% 1.65k/1.65k [00:00<00:00, 5.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1yP2dBE8K9QtUiaLPewtQRr-hDH0gOgfa
To: /content/list_street.txt
100% 6.65k/6.65k [00:00<00:00, 14.3MB/s]


In [68]:
def _judge_normalize(s: str, context=None):
    return s

EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json') as f:
    data = json.load(f)

with open("list_province.txt") as f:
    list_province = f.read().strip().splitlines()

with open("list_ward.txt") as f:
    list_ward = f.read().strip().splitlines()

with open("list_street.txt") as f:
    list_street = f.read().strip().splitlines()

summary_only = True
df = []
solution = Solution(
    list_province=list_province,
    list_ward=list_ward,
    list_street=list_street,
)
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["address"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = _judge_normalize(answer["province"])
        answer["ward_normalized"] = _judge_normalize(answer["ward"])
        answer["street_normalized"] = _judge_normalize(answer["street"])

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = _judge_normalize(result["province"])
        result["ward_normalized"] = _judge_normalize(result["ward"])
        result["street_normalized"] = _judge_normalize(result["street"])

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        street_correct = int(answer["street_normalized"] == result["street_normalized"])
        ok = province_correct + street_correct + ward_correct

        df.append([
            test_idx,
            address,

            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,

            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,

            answer["street"],
            result["street"],
            answer["street_normalized"],
            result["street_normalized"],
            street_correct,

            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,

            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,

            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,

            answer["street"],
            "EXCEPTION",
            answer["street_normalized"],
            "EXCEPTION",
            0,

            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',

    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',

    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',

    'street',
    'street_student',
    'street_normalized',
    'street_student_normalized',
    'street_correct',

    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()

Generated variants for street '1': {'duong1', '1st', 'd1', '1street', '1'}
Generated variants for street '12': {'duong12', 'd12', '12st', '12street', '12'}
Generated variants for street '2thang9': {'2thang9', 'd2thang9', '2thang9street', 'duong2thang9', '2thang9st'}
Generated variants for street '3thang2': {'duong3thang2', '3thang2street', 'd3thang2', '3thang2', '3thang2st'}
Generated variants for street 'ancu7': {'dancu7', 'ancu7st', 'ancu7', 'ancu7street', 'duongancu7'}
Generated variants for street 'anduongvuong': {'danduongvuong', 'anduongvuong', 'anduongvuongst', 'anduongvuongstreet', 'duonganduongvuong'}
Generated variants for street 'anninh': {'anninh', 'anninhst', 'anninhstreet', 'duonganninh', 'danninh'}
Generated variants for street 'anthuong23': {'danthuong23', 'anthuong23st', 'anthuong23', 'duonganthuong23', 'anthuong23street'}
Generated variants for street 'anthuong33': {'anthuong33st', 'duonganthuong33', 'danthuong33', 'anthuong33street', 'anthuong33'}
Generated variants 